# 🚀 Google Colab × Antigravity 智能联动实验室

本 Notebook 包含：
1. **Cloudflare Quick Tunnel 远程 SSH 穿透**：一键生成终端直连通道，让本地助手可直接操控云端 GPU。
2. **经典深度学习与金融量化实验**：PyTorch GPU 张量加速测试、多周期量价特征工程与 LightGBM 拟合。
3. **云端音视频处理套件测试**：FFmpeg 环境与 AI 语音转字幕。

## 步骤 1：启动 Cloudflare Tunnel 与 SSH 服务
点击运行下方代码，等待数秒即可看到生成的 SSH 直连命令。

In [ ]:
# 1. 一键启动稳定版 Colab-SSH (基于 Cloudflare Tunnel)
!pip install colab_ssh --upgrade -q
from colab_ssh import launch_ssh_cloudflared
launch_ssh_cloudflared(password="colab123")


## 步骤 2：经典机器学习与金融多周期量价实验
测试 GPU 矩阵算力以及基于多周期形态（如均线翻转斜率与量能变化）的分类与夏普回测。

In [ ]:
import torch, time
import numpy as np

# 1. GPU 检测
print("PyTorch:", torch.__version__)
print("CUDA 可用:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU 显卡型号:", torch.cuda.get_device_name(0))
    print(f"GPU 显存大小: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

# 2. 深度学习神经网络梯度反向传播
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
x = torch.randn(4096, 128, device=device)
y = torch.randn(4096, 1, device=device)
model = torch.nn.Sequential(
    torch.nn.Linear(128, 256),
    torch.nn.ReLU(),
    torch.nn.Linear(256, 1)
).to(device)
loss_fn = torch.nn.MSELoss()
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

t0 = time.time()
for _ in range(100):
    loss = loss_fn(model(x), y)
    opt.zero_grad()
    loss.backward()
    opt.step()
print(f"✅ 深度学习 100 Epochs 训练完成，耗时: {time.time()-t0:.4f} 秒")

# 3. 金融多周期量价形态与 GBDT 模拟
try:
    import lightgbm as lgb
    np.random.seed(42)
    n = 10000
    ma_slope = np.random.randn(n)        # 60分均线斜率
    vol_ratio = np.random.exponential(1.0, n)  # 突破放量倍数
    bias = np.random.randn(n)            # 15分乖离率
    y = ((0.12 * ma_slope + 0.08 * vol_ratio + np.random.normal(0, 0.5, n)) > 0).astype(int)
    X = np.column_stack([ma_slope, vol_ratio, bias])
    clf = lgb.LGBMClassifier(n_estimators=50, max_depth=3, verbose=-1)
    clf.fit(X[:8000], y[:8000])
    acc = np.mean(clf.predict(X[8000:]) == y[8000:])
    print(f"✅ 金融 LightGBM 因子重要性: {clf.feature_importances_} | 测试集准确率: {acc*100:.2f}%")
except Exception as e:
    print("LightGBM 运行跳过:", e)

## 步骤 3：微软 Qlib 工业级 AI 量化金融平台实验
使用真实 A 股历史行情数据、Alpha158 量价因子库进行时序特征提取与模型训练，并评估真实 IC / Rank IC。

In [ ]:
# 1. 安装与诊断 Qlib 环境
import sys, subprocess
print("当前 Python 版本:", sys.version)

cmd = "pip install --upgrade pyqlib tables lightgbm"
res = subprocess.run(cmd, shell=True, capture_output=True, text=True)
print("pip install 状态码:", res.returncode)
if res.returncode != 0:
    print("pip install 错误输出:", res.stderr[-500:])
else:
    print("✅ pyqlib 安装成功！")

# 2. 尝试导入并下载数据
try:
    import qlib
    print("✅ Qlib 版本:", qlib.__version__)
    get_data_cmd = "python -m qlib.run.get_data qlib_data --target_dir ~/.qlib/qlib_data/cn_data --region cn --exists_skip True"
    d_res = subprocess.run(get_data_cmd, shell=True, capture_output=True, text=True)
    print("数据下载状态码:", d_res.returncode)
    if d_res.returncode != 0:
        print("数据下载日志:", d_res.stdout[-300:], d_res.stderr[-300:])
    else:
        print("✅ A股数据集下载成功！")
except Exception as e:
    print("❌ 导入或运行异常:", e)


In [ ]:
# 4. 运行微软 Alpha158 真实因子流水线与模型评测
from qlib.contrib.data.handler import Alpha158
from qlib.contrib.model.gbdt import LGBModel
from qlib.utils import init_instance_by_config
import pandas as pd
import numpy as np

print("🔄 正在构建 Alpha158 因子 (覆盖动量、波动率、均线斜率、量价背离等 158 个金融因子)...")
dataset_config = {
    "class": "DatasetH",
    "module_path": "qlib.data.dataset",
    "kwargs": {
        "handler": {
            "class": "Alpha158",
            "module_path": "qlib.contrib.data.handler",
            "kwargs": {
                "start_time": "2019-01-01",
                "end_time": "2020-06-30",
                "fit_start_time": "2019-01-01",
                "fit_end_time": "2019-12-31",
                "instruments": "csi300",
            },
        },
        "segments": {
            "train": ("2019-01-01", "2019-12-31"),
            "test": ("2020-01-01", "2020-06-30"),
        },
    },
}

dataset = init_instance_by_config(dataset_config)
df_train = dataset.prepare("train")
df_test = dataset.prepare("test")
print(f"✅ 训练集样本数: {len(df_train)}, 测试集样本数: {len(df_test)}")
print(f"✅ 特征维度数: {df_train.shape[1]} (Alpha158)")

# 训练 LightGBM 因子预测模型
model = LGBModel(loss="mse", n_estimators=60, learning_rate=0.08, verbose=-1)
print("🚀 训练模型中...")
model.fit(dataset)

# 预测与计算 IC 表现
pred = model.predict(dataset)
label = dataset.prepare("test", col_set="label")
pred_label = pd.concat([pred, label], axis=1).dropna()

ic = pred_label.groupby(level="datetime").apply(lambda x: x.iloc[:, 0].corr(x.iloc[:, 1])).mean()
rank_ic = pred_label.groupby(level="datetime").apply(lambda x: x.iloc[:, 0].corr(x.iloc[:, 1], method="spearman")).mean()

print("=" * 60)
print("🏆 微软 Qlib A 股 Alpha158 实盘历史回测指标:")
print(f"  测试集样本外信息系数 (Normal IC) : {ic:.4f}")
print(f"  测试集样本外秩相关系数 (Rank IC)   : {rank_ic:.4f}")
if rank_ic > 0.03:
    print("  💡 评语: Rank IC > 0.03，具备显著的真实超额选股能力 (Alpha)！")
print("=" * 60)